# 10-Fold Baselines: BiomedCLIP Only and BiomedCLIP + MLP

This notebook evaluates two discriminative baselines on the same 14-label chest X-ray split file used by the FAISS/diffusion/GNN experiments.

Methods:

- `biomedclip_zero_shot`: frozen BiomedCLIP image/text similarity with present-vs-absent prompts for each label.
- `biomedclip_mlp_14`: frozen BiomedCLIP image embeddings with a supervised MLP head for the 14 labels.

Outputs are saved under `/data/liangz2/openi/multi_kg/biomedclip_10fold_baselines` when the cloud root exists; otherwise they are saved under the current working directory.

## 0. Optional Dependencies

In [1]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "open_clip_torch", "transformers", "pandas", "numpy", "pillow", "tqdm", "scikit-learn",
    ])

## 1. Imports and Configuration

In [2]:
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

DATA_ROOT = Path("/data/liangz2/openi/multi_kg") if Path("/data/liangz2/openi/multi_kg").exists() else Path.cwd()
OUTPUT_ROOT = DATA_ROOT / "biomedclip_10fold_baselines"
EMBEDDING_DIR = OUTPUT_ROOT / "biomedclip_embeddings"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_CSV_CANDIDATES = [
    DATA_ROOT / "multi_kg_image_json_10fold.csv",
    Path("/data/liangz2/openi/inverse_diff/multi_kg_image_json_10fold.csv"),
    Path("/Users/liangz2/Documents/inverse_diffusion/multi_kg_image_json_10fold.csv"),
]
SPLIT_CSV = next((p for p in SPLIT_CSV_CANDIDATES if p.exists()), SPLIT_CSV_CANDIDATES[0])

BIOMEDCLIP_REPO = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

FOLD_INDICES = list(range(10))
VAL_FRAC = 0.10
SKIP_COMPLETED = True
FORCE_RERUN_FOLDS = []
SAVE_PREDICTIONS = True

# Embedding extraction.
IMAGE_BATCH_SIZE = 64
TEXT_BATCH_SIZE = 64
EMBEDDINGS_PATH = EMBEDDING_DIR / "multi_kg_10fold_biomedclip_image_embeddings.npy"
EMBEDDING_METADATA_PATH = EMBEDDING_DIR / "multi_kg_10fold_biomedclip_embedding_metadata.csv"
EMBEDDING_MANIFEST_PATH = EMBEDDING_DIR / "multi_kg_10fold_biomedclip_embedding_manifest.json"
RECOMPUTE_EMBEDDINGS = False

# MLP head training.
MLP_HIDDEN_DIMS = (512,)
MLP_DROPOUT = 0.20
MLP_EPOCHS = 30
MLP_BATCH_SIZE = 512
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_PATIENCE = 6
GRAD_CLIP_NORM = 5.0

SCORING_LABELS = (
    "normal",
    "enlarged cardiomediastinum", "cardiomegaly", "lung opacity", "lung lesion", "edema",
    "consolidation", "pneumonia", "atelectasis", "pneumothorax",
    "pleural effusion", "pleural other", "fracture", "support devices",
)
FINDING_LABELS = SCORING_LABELS[1:]
TARGET_COLUMNS = ["y_" + label.replace(" ", "_") for label in SCORING_LABELS]

print("DEVICE", DEVICE, "DTYPE", DTYPE)
print("DATA_ROOT", DATA_ROOT)
print("OUTPUT_ROOT", OUTPUT_ROOT)
print("SPLIT_CSV", SPLIT_CSV)

DEVICE cuda DTYPE torch.float16
DATA_ROOT /data/liangz2/openi/multi_kg
OUTPUT_ROOT /data/liangz2/openi/multi_kg/biomedclip_10fold_baselines
SPLIT_CSV /data/liangz2/openi/multi_kg/multi_kg_image_json_10fold.csv


## 2. Metrics and File Helpers

In [3]:
def method_slug(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


def ensure_fold_dirs(fold_index: int) -> Dict[str, Path]:
    fold_dir = OUTPUT_ROOT / f"fold{fold_index}"
    dirs = {
        "fold": fold_dir,
        "models": fold_dir / "models",
        "metrics": fold_dir / "metrics",
        "predictions": fold_dir / "predictions",
        "logs": fold_dir / "logs",
        "tuning": fold_dir / "tuning",
        "cache": fold_dir / "cache",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def sigmoid_np(logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-logits))


def evaluate_prob_matrix(probs: np.ndarray, targets: np.ndarray, thresholds: Optional[np.ndarray] = None) -> pd.DataFrame:
    y = targets.astype(int)
    if thresholds is None:
        thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    rows = []
    for j, label in enumerate(SCORING_LABELS):
        if len(np.unique(y[:, j])) < 2:
            continue
        pred = (probs[:, j] >= thresholds[j]).astype(int)
        rows.append({
            "label": label,
            "auroc": roc_auc_score(y[:, j], probs[:, j]),
            "auprc": average_precision_score(y[:, j], probs[:, j]),
            "f1": f1_score(y[:, j], pred, zero_division=0),
            "threshold": float(thresholds[j]),
            "support": int(y[:, j].sum()),
        })
    return pd.DataFrame(rows)


def metric_summary(metrics: pd.DataFrame, fold_index: int, method: str, extra: Optional[Dict] = None) -> Dict:
    disease = metrics[metrics["label"] != "normal"] if not metrics.empty else metrics
    row = {
        "fold": int(fold_index),
        "method": method,
        "disease_macro_auroc": float(disease["auroc"].mean()) if len(disease) else float("nan"),
        "disease_macro_auprc": float(disease["auprc"].mean()) if len(disease) else float("nan"),
        "disease_macro_f1": float(disease["f1"].mean()) if len(disease) else float("nan"),
        "all_macro_auroc": float(metrics["auroc"].mean()) if len(metrics) else float("nan"),
        "all_macro_auprc": float(metrics["auprc"].mean()) if len(metrics) else float("nan"),
        "all_macro_f1": float(metrics["f1"].mean()) if len(metrics) else float("nan"),
        "num_scored_labels": int(len(metrics)),
    }
    if extra:
        row.update(extra)
    return row


def tune_f1_thresholds(probs: np.ndarray, targets: np.ndarray, grid: Optional[np.ndarray] = None) -> np.ndarray:
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    y = targets.astype(int)
    for j in range(probs.shape[1]):
        if len(np.unique(y[:, j])) < 2:
            continue
        best_f1, best_thr = -1.0, 0.5
        for thr in grid:
            pred = (probs[:, j] >= thr).astype(int)
            value = f1_score(y[:, j], pred, zero_division=0)
            if value > best_f1:
                best_f1, best_thr = value, float(thr)
        thresholds[j] = best_thr
    return thresholds


def save_json(obj: Dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def save_method_metrics(fold_index: int, method: str, metrics: pd.DataFrame, fold_dirs: Dict[str, Path], threshold_mode: str) -> Dict:
    suffix = "test_metrics" if threshold_mode == "fixed_0.5" else "test_metrics_val_tuned_thresholds"
    metrics_path = fold_dirs["metrics"] / f"{method}_{suffix}.csv"
    metrics.to_csv(metrics_path, index=False)
    return metric_summary(metrics, fold_index, method, extra={
        "threshold_mode": threshold_mode,
        "metrics_path": str(metrics_path),
    })


def save_predictions(path: Path, frame: pd.DataFrame, probs: np.ndarray, targets: np.ndarray, block_name: str) -> pd.DataFrame:
    rows = []
    for i, row in frame.reset_index(drop=True).iterrows():
        out = {
            "study_id": str(row.get("study_id", "")),
            "subject_id": str(row.get("subject_id", "")),
            "image_path": str(row.get("image_path", "")),
            "resolved_image_path": str(row.get("resolved_image_path", "")),
        }
        for j, label in enumerate(SCORING_LABELS):
            out[f"{block_name}_prob_{label}"] = float(probs[i, j])
            out[f"target_{label}"] = int(targets[i, j] >= 0.5)
        rows.append(out)
    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(path, index=False)
    return pred_df

## 3. Load 10-Fold Split Table

In [4]:
def resolve_image_path(path_value: str, row: Optional[pd.Series] = None) -> str:
    raw = Path(str(path_value))
    candidates = [raw]
    raw_str = str(raw)
    replacements = [
        ("/vf/users/liangz2/openi/multi_kg", "/data/liangz2/openi/multi_kg"),
        ("/data/liangz2/openi/multi_kg", "/vf/users/liangz2/openi/multi_kg"),
    ]
    for old, new in replacements:
        if raw_str.startswith(old):
            candidates.append(Path(raw_str.replace(old, new, 1)))
    if row is not None:
        image_name = str(row.get("image_name", Path(raw_str).name))
        original_split = str(row.get("original_split", "")).strip()
        if original_split:
            candidates.append(DATA_ROOT / original_split / image_name)
        candidates.append(DATA_ROOT / "train" / image_name)
        candidates.append(DATA_ROOT / "test" / image_name)
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return str(candidates[0])


def load_split_table() -> pd.DataFrame:
    if not SPLIT_CSV.exists():
        raise FileNotFoundError(f"Split CSV not found. Checked: {SPLIT_CSV_CANDIDATES}")
    frame = pd.read_csv(SPLIT_CSV)
    required = ["study_id", "image_path"] + TARGET_COLUMNS + [f"fold_{i}" for i in FOLD_INDICES]
    missing = [col for col in required if col not in frame.columns]
    if missing:
        raise ValueError(f"Split CSV is missing required columns: {missing}")
    frame = frame.copy()
    frame["study_id"] = frame["study_id"].astype(str)
    if "subject_id" in frame.columns:
        frame["subject_id"] = frame["subject_id"].astype(str)
    frame["image_path"] = frame["image_path"].astype(str)
    if "image_name" not in frame.columns:
        frame["image_name"] = frame["image_path"].map(lambda x: Path(str(x)).name)
    frame["resolved_image_path"] = [resolve_image_path(row["image_path"], row) for _, row in frame.iterrows()]
    frame["target"] = frame[TARGET_COLUMNS].astype(np.float32).values.tolist()
    frame["row_id"] = np.arange(len(frame), dtype=int)
    return frame


def verify_image_paths(frame: pd.DataFrame) -> None:
    exists = frame["resolved_image_path"].map(lambda x: Path(str(x)).exists())
    if not exists.all():
        missing = frame.loc[~exists, ["row_id", "study_id", "image_path", "resolved_image_path"]]
        out = OUTPUT_ROOT / "missing_image_paths.csv"
        missing.to_csv(out, index=False)
        raise FileNotFoundError(f"{len(missing)} image paths are missing. Saved details to {out}")
    print("All image paths exist:", len(frame))


def make_train_val_split(frame: pd.DataFrame, fold_index: int, val_frac: float = VAL_FRAC) -> Tuple[pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1.0, random_state=SEED + int(fold_index)).reset_index(drop=True)
    n_val = max(1, int(round(len(shuffled) * val_frac))) if len(shuffled) else 0
    val = shuffled.iloc[:n_val].reset_index(drop=True)
    train = shuffled.iloc[n_val:].reset_index(drop=True)
    return train, val


def load_fold_frames(frame: pd.DataFrame, fold_index: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    fold_col = f"fold_{fold_index}"
    train_full = frame[frame[fold_col].astype(str).str.lower().eq("train")].reset_index(drop=True)
    test = frame[frame[fold_col].astype(str).str.lower().eq("test")].reset_index(drop=True)
    train_part, val = make_train_val_split(train_full, fold_index)
    return train_full, train_part, val, test


split_table_df = load_split_table()
print("split rows", len(split_table_df))
print("columns", len(split_table_df.columns))
print(split_table_df[["study_id", "image_path", "resolved_image_path", "fold_0"]].head())
verify_image_paths(split_table_df)

split rows 44100
columns 39
   study_id                                         image_path  \
0  50003542  /vf/users/liangz2/openi/multi_kg/train/5000354...   
1  50003755  /vf/users/liangz2/openi/multi_kg/train/5000375...   
2  50008255  /vf/users/liangz2/openi/multi_kg/train/5000825...   
3  50010166  /vf/users/liangz2/openi/multi_kg/train/5001016...   
4  50010229  /vf/users/liangz2/openi/multi_kg/train/5001022...   

                                 resolved_image_path fold_0  
0  /vf/users/liangz2/openi/multi_kg/train/5000354...   test  
1  /vf/users/liangz2/openi/multi_kg/train/5000375...   test  
2  /vf/users/liangz2/openi/multi_kg/train/5000825...   test  
3  /vf/users/liangz2/openi/multi_kg/train/5001016...   test  
4  /vf/users/liangz2/openi/multi_kg/train/5001022...   test  
All image paths exist: 44100


## 4. BiomedCLIP Model and Embedding Cache

In [5]:
biomedclip_model = None
biomedclip_preprocess = None
biomedclip_tokenizer = None


def get_biomedclip_model():
    global biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer
    if biomedclip_model is None:
        from open_clip import create_model_from_pretrained, get_tokenizer
        model, preprocess = create_model_from_pretrained(BIOMEDCLIP_REPO)
        model = model.to(DEVICE)
        model.eval()
        for param in model.parameters():
            param.requires_grad = False
        tokenizer = get_tokenizer(BIOMEDCLIP_REPO)
        biomedclip_model = model
        biomedclip_preprocess = preprocess
        biomedclip_tokenizer = tokenizer
    return biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer


def load_image_for_biomedclip(path: str) -> Image.Image:
    return Image.open(path).convert("RGB")


@torch.no_grad()
def encode_images(paths: Sequence[str], batch_size: int = IMAGE_BATCH_SIZE) -> np.ndarray:
    model, preprocess, _ = get_biomedclip_model()
    chunks = []
    for start in tqdm(range(0, len(paths), batch_size), desc="BiomedCLIP image embeddings"):
        batch_paths = paths[start:start + batch_size]
        images = [preprocess(load_image_for_biomedclip(path)) for path in batch_paths]
        pixels = torch.stack(images).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            feats = model.encode_image(pixels)
        feats = F.normalize(feats.float(), dim=-1)
        chunks.append(feats.cpu().numpy().astype("float32"))
    return np.concatenate(chunks, axis=0) if chunks else np.empty((0, 512), dtype="float32")


@torch.no_grad()
def encode_texts(texts: Sequence[str], batch_size: int = TEXT_BATCH_SIZE) -> np.ndarray:
    model, _, tokenizer = get_biomedclip_model()
    chunks = []
    for start in range(0, len(texts), batch_size):
        batch = list(texts[start:start + batch_size])
        tokens = tokenizer(batch).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            feats = model.encode_text(tokens)
        feats = F.normalize(feats.float(), dim=-1)
        chunks.append(feats.cpu().numpy().astype("float32"))
    return np.concatenate(chunks, axis=0) if chunks else np.empty((0, 512), dtype="float32")


def embedding_cache_is_valid(frame: pd.DataFrame) -> bool:
    if RECOMPUTE_EMBEDDINGS:
        return False
    if not EMBEDDINGS_PATH.exists() or not EMBEDDING_METADATA_PATH.exists():
        return False
    meta = pd.read_csv(EMBEDDING_METADATA_PATH)
    if len(meta) != len(frame):
        return False
    if "row_id" not in meta.columns or not np.array_equal(meta["row_id"].to_numpy(), frame["row_id"].to_numpy()):
        return False
    if "study_id" in meta.columns and not np.array_equal(meta["study_id"].astype(str).to_numpy(), frame["study_id"].astype(str).to_numpy()):
        return False
    return True


def get_or_create_image_embeddings(frame: pd.DataFrame) -> np.ndarray:
    if embedding_cache_is_valid(frame):
        print("Loading cached embeddings:", EMBEDDINGS_PATH)
        return np.load(EMBEDDINGS_PATH).astype("float32")
    paths = frame["resolved_image_path"].astype(str).tolist()
    embeddings = encode_images(paths, batch_size=IMAGE_BATCH_SIZE)
    np.save(EMBEDDINGS_PATH, embeddings.astype("float32"))
    meta_cols = [col for col in ["row_id", "study_id", "subject_id", "image_path", "resolved_image_path", "image_name"] if col in frame.columns]
    frame[meta_cols].to_csv(EMBEDDING_METADATA_PATH, index=False)
    save_json({
        "biomedclip_repo": BIOMEDCLIP_REPO,
        "split_csv": str(SPLIT_CSV),
        "num_rows": int(len(frame)),
        "embedding_dim": int(embeddings.shape[1]),
        "embeddings_path": str(EMBEDDINGS_PATH),
        "metadata_path": str(EMBEDDING_METADATA_PATH),
    }, EMBEDDING_MANIFEST_PATH)
    print("Saved embeddings:", EMBEDDINGS_PATH, embeddings.shape)
    return embeddings.astype("float32")


image_embeddings = get_or_create_image_embeddings(split_table_df)
print("image_embeddings", image_embeddings.shape)

BiomedCLIP image embeddings:   0%|          | 0/690 [00:00<?, ?it/s]

Saved embeddings: /data/liangz2/openi/multi_kg/biomedclip_10fold_baselines/biomedclip_embeddings/multi_kg_10fold_biomedclip_image_embeddings.npy (44100, 512)
image_embeddings (44100, 512)


## 5. BiomedCLIP Zero-Shot Prompt Scoring

For each label, the zero-shot score is computed as a present-vs-absent softmax. This gives one probability per label and avoids forcing the 14 labels to compete with each other.

In [6]:
def positive_prompts(label: str) -> List[str]:
    if label == "normal":
        return [
            "a normal frontal chest x-ray.",
            "a chest radiograph with no acute cardiopulmonary abnormality.",
            "a normal chest radiograph without focal disease.",
        ]
    return [
        f"a frontal chest x-ray showing {label}.",
        f"a chest radiograph with evidence of {label}.",
        f"radiographic finding of {label} on chest x-ray.",
    ]


def negative_prompts(label: str) -> List[str]:
    if label == "normal":
        return [
            "an abnormal frontal chest x-ray.",
            "a chest radiograph with cardiopulmonary abnormality.",
            "a chest x-ray showing abnormal findings.",
        ]
    return [
        f"a frontal chest x-ray without {label}.",
        f"no radiographic evidence of {label}.",
        f"a chest radiograph negative for {label}.",
    ]


def mean_text_embedding(prompts: Sequence[str]) -> np.ndarray:
    emb = encode_texts(list(prompts), batch_size=TEXT_BATCH_SIZE)
    mean = emb.mean(axis=0, keepdims=True)
    mean = mean / np.maximum(np.linalg.norm(mean, axis=1, keepdims=True), 1e-12)
    return mean.astype("float32")[0]


def build_zero_shot_text_embeddings() -> Tuple[np.ndarray, np.ndarray]:
    pos, neg = [], []
    for label in SCORING_LABELS:
        pos.append(mean_text_embedding(positive_prompts(label)))
        neg.append(mean_text_embedding(negative_prompts(label)))
    return np.stack(pos).astype("float32"), np.stack(neg).astype("float32")


def biomedclip_logit_scale() -> float:
    model, _, _ = get_biomedclip_model()
    if hasattr(model, "logit_scale"):
        return float(model.logit_scale.exp().detach().cpu())
    return 100.0


def zero_shot_probs(image_emb: np.ndarray, pos_text_emb: np.ndarray, neg_text_emb: np.ndarray) -> np.ndarray:
    scale = biomedclip_logit_scale()
    pos_logits = scale * (image_emb.astype("float32") @ pos_text_emb.T)
    neg_logits = scale * (image_emb.astype("float32") @ neg_text_emb.T)
    pair = np.stack([neg_logits, pos_logits], axis=-1)
    pair = pair - pair.max(axis=-1, keepdims=True)
    exp_pair = np.exp(pair)
    probs = exp_pair[..., 1] / exp_pair.sum(axis=-1)
    return probs.astype("float32")


ZS_POS_TEXT_EMB, ZS_NEG_TEXT_EMB = build_zero_shot_text_embeddings()
print("zero-shot text embeddings", ZS_POS_TEXT_EMB.shape, ZS_NEG_TEXT_EMB.shape)

zero-shot text embeddings (14, 512) (14, 512)


## 6. MLP Head Training

In [7]:
class BiomedClipMLPHead(nn.Module):
    def __init__(self, input_dim: int, num_outputs: int, hidden_dims: Sequence[int] = MLP_HIDDEN_DIMS, dropout: float = MLP_DROPOUT):
        super().__init__()
        layers = []
        prev = int(input_dim)
        for hidden in hidden_dims:
            hidden = int(hidden)
            if hidden <= 0:
                continue
            layers.extend([nn.Linear(prev, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout)])
            prev = hidden
        layers.append(nn.Linear(prev, num_outputs))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.float())


@torch.no_grad()
def predict_mlp_logits(head: nn.Module, embeddings: np.ndarray, batch_size: int = 8192) -> np.ndarray:
    head.eval()
    chunks = []
    for start in range(0, len(embeddings), batch_size):
        x = torch.from_numpy(embeddings[start:start + batch_size]).to(DEVICE).float()
        chunks.append(head(x).detach().cpu())
    return torch.cat(chunks).numpy().astype("float32") if chunks else np.empty((0, len(SCORING_LABELS)), dtype="float32")


def pos_weight_from_targets(y: np.ndarray) -> torch.Tensor:
    target = torch.from_numpy(y.astype("float32"))
    pos = target.sum(dim=0)
    neg = target.size(0) - pos
    return torch.clamp(neg / torch.clamp(pos, min=1.0), min=1.0, max=50.0)


def train_mlp_head(
    train_x: np.ndarray,
    train_y: np.ndarray,
    val_x: np.ndarray,
    val_y: np.ndarray,
    fold_index: int,
    fold_dirs: Dict[str, Path],
) -> Tuple[nn.Module, pd.DataFrame, Dict]:
    head = BiomedClipMLPHead(train_x.shape[1], len(SCORING_LABELS)).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_from_targets(train_y).to(DEVICE))
    dataset = TensorDataset(torch.from_numpy(train_x.astype("float32")), torch.from_numpy(train_y.astype("float32")))
    loader = DataLoader(dataset, batch_size=MLP_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda"))

    best_score = -float("inf")
    best_state = None
    best_epoch = -1
    stale = 0
    logs = []

    for epoch in range(MLP_EPOCHS):
        head.train()
        losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE).float()
            yb = yb.to(DEVICE).float()
            optimizer.zero_grad(set_to_none=True)
            logits = head(xb)
            loss = criterion(logits, yb)
            loss.backward()
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        train_probs = sigmoid_np(predict_mlp_logits(head, train_x))
        val_probs = sigmoid_np(predict_mlp_logits(head, val_x))
        train_metrics = evaluate_prob_matrix(train_probs, train_y)
        val_metrics = evaluate_prob_matrix(val_probs, val_y)
        train_auc = float(train_metrics[train_metrics["label"] != "normal"]["auroc"].mean())
        val_auc = float(val_metrics[val_metrics["label"] != "normal"]["auroc"].mean())
        row = {
            "fold": int(fold_index),
            "epoch": int(epoch),
            "loss": float(np.mean(losses)) if losses else float("nan"),
            "train_disease_macro_auroc": train_auc,
            "val_disease_macro_auroc": val_auc,
        }
        logs.append(row)
        print(f"fold={fold_index} epoch={epoch} loss={row['loss']:.4f} val_disease_macro_auroc={val_auc:.4f}")

        if np.isfinite(val_auc) and val_auc > best_score:
            best_score = val_auc
            best_epoch = int(epoch)
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= MLP_PATIENCE:
            print(f"fold={fold_index}: early stopping at epoch {epoch}")
            break

    if best_state is not None:
        head.load_state_dict(best_state)
    log_df = pd.DataFrame(logs)
    log_path = fold_dirs["logs"] / "biomedclip_mlp_14_training_log.csv"
    log_df.to_csv(log_path, index=False)
    model_path = fold_dirs["models"] / f"biomedclip_mlp_14_fold{fold_index}_seed{SEED}.pt"
    checkpoint = {
        "state_dict": head.state_dict(),
        "fold": int(fold_index),
        "seed": int(SEED),
        "labels": list(SCORING_LABELS),
        "input_dim": int(train_x.shape[1]),
        "hidden_dims": list(MLP_HIDDEN_DIMS),
        "dropout": float(MLP_DROPOUT),
        "best_epoch": int(best_epoch),
        "best_val_disease_macro_auroc": float(best_score),
        "biomedclip_repo": BIOMEDCLIP_REPO,
    }
    torch.save(checkpoint, model_path)
    return head, log_df, {"model_path": str(model_path), "best_epoch": best_epoch, "best_val_disease_macro_auroc": best_score}

## 7. Per-Fold Runner

In [8]:
def targets_from_frame(frame: pd.DataFrame) -> np.ndarray:
    return frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)


def embeddings_from_frame(frame: pd.DataFrame) -> np.ndarray:
    return image_embeddings[frame["row_id"].to_numpy(dtype=int)].astype("float32")


def fold_is_complete(fold_index: int, fold_dirs: Optional[Dict[str, Path]] = None) -> bool:
    fold_dirs = ensure_fold_dirs(fold_index) if fold_dirs is None else fold_dirs
    return (
        (fold_dirs["fold"] / "FOLD_COMPLETE.json").exists()
        and (fold_dirs["metrics"] / "fold_method_summary.csv").exists()
        and (fold_dirs["metrics"] / "fold_label_metrics.csv").exists()
    )


def load_completed_fold(fold_dirs: Dict[str, Path]) -> Dict[str, pd.DataFrame]:
    return {
        "summary": pd.read_csv(fold_dirs["metrics"] / "fold_method_summary.csv"),
        "label_metrics": pd.read_csv(fold_dirs["metrics"] / "fold_label_metrics.csv"),
        "training_logs": pd.read_csv(fold_dirs["logs"] / "biomedclip_mlp_14_training_log.csv") if (fold_dirs["logs"] / "biomedclip_mlp_14_training_log.csv").exists() else pd.DataFrame(),
    }


def run_one_fold(fold_index: int) -> Dict[str, pd.DataFrame]:
    fold_dirs = ensure_fold_dirs(fold_index)
    if SKIP_COMPLETED and fold_index not in FORCE_RERUN_FOLDS and fold_is_complete(fold_index, fold_dirs):
        print(f"fold {fold_index}: complete, loading saved results")
        return load_completed_fold(fold_dirs)

    train_full, train_part, val, test = load_fold_frames(split_table_df, fold_index)
    print(f"\n===== Fold {fold_index} =====")
    print("train_full", len(train_full), "train_part", len(train_part), "val", len(val), "test", len(test))

    train_x = embeddings_from_frame(train_part)
    val_x = embeddings_from_frame(val)
    test_x = embeddings_from_frame(test)
    train_y = targets_from_frame(train_part)
    val_y = targets_from_frame(val)
    test_y = targets_from_frame(test)

    summary_rows = []
    label_tables = []

    # Method 1: BiomedCLIP zero-shot present-vs-absent prompts.
    val_zs_probs = zero_shot_probs(val_x, ZS_POS_TEXT_EMB, ZS_NEG_TEXT_EMB)
    test_zs_probs = zero_shot_probs(test_x, ZS_POS_TEXT_EMB, ZS_NEG_TEXT_EMB)
    zs_thresholds = tune_f1_thresholds(val_zs_probs, val_y)
    pd.DataFrame({"label": SCORING_LABELS, "threshold": zs_thresholds}).to_csv(fold_dirs["tuning"] / "biomedclip_zero_shot_val_f1_thresholds.csv", index=False)
    zs_metrics = evaluate_prob_matrix(test_zs_probs, test_y)
    zs_metrics_tuned = evaluate_prob_matrix(test_zs_probs, test_y, thresholds=zs_thresholds)
    summary_rows.append(save_method_metrics(fold_index, "biomedclip_zero_shot", zs_metrics, fold_dirs, "fixed_0.5"))
    summary_rows.append(save_method_metrics(fold_index, "biomedclip_zero_shot", zs_metrics_tuned, fold_dirs, "val_tuned_f1"))
    label_fixed = zs_metrics.copy(); label_fixed.insert(0, "threshold_mode", "fixed_0.5"); label_fixed.insert(0, "method", "biomedclip_zero_shot"); label_fixed.insert(0, "fold", fold_index); label_tables.append(label_fixed)
    label_tuned = zs_metrics_tuned.copy(); label_tuned.insert(0, "threshold_mode", "val_tuned_f1"); label_tuned.insert(0, "method", "biomedclip_zero_shot"); label_tuned.insert(0, "fold", fold_index); label_tables.append(label_tuned)
    if SAVE_PREDICTIONS:
        save_predictions(fold_dirs["predictions"] / "biomedclip_zero_shot_test_predictions.csv", test, test_zs_probs, test_y, "zero_shot")

    # Method 2: Frozen BiomedCLIP image embeddings + MLP head.
    head, train_log, ckpt_extra = train_mlp_head(train_x, train_y, val_x, val_y, fold_index, fold_dirs)
    train_mlp_probs = sigmoid_np(predict_mlp_logits(head, train_x))
    val_mlp_probs = sigmoid_np(predict_mlp_logits(head, val_x))
    test_mlp_probs = sigmoid_np(predict_mlp_logits(head, test_x))
    mlp_thresholds = tune_f1_thresholds(val_mlp_probs, val_y)
    pd.DataFrame({"label": SCORING_LABELS, "threshold": mlp_thresholds}).to_csv(fold_dirs["tuning"] / "biomedclip_mlp_14_val_f1_thresholds.csv", index=False)

    train_mlp_metrics = evaluate_prob_matrix(train_mlp_probs, train_y)
    val_mlp_metrics = evaluate_prob_matrix(val_mlp_probs, val_y)
    test_mlp_metrics = evaluate_prob_matrix(test_mlp_probs, test_y)
    test_mlp_metrics_tuned = evaluate_prob_matrix(test_mlp_probs, test_y, thresholds=mlp_thresholds)
    train_mlp_metrics.to_csv(fold_dirs["metrics"] / "biomedclip_mlp_14_train_metrics.csv", index=False)
    val_mlp_metrics.to_csv(fold_dirs["metrics"] / "biomedclip_mlp_14_val_metrics.csv", index=False)

    extra = {**ckpt_extra, "threshold_mode": "fixed_0.5"}
    summary_rows.append(save_method_metrics(fold_index, "biomedclip_mlp_14", test_mlp_metrics, fold_dirs, "fixed_0.5"))
    summary_rows[-1].update(extra)
    extra_tuned = {**ckpt_extra, "threshold_mode": "val_tuned_f1"}
    summary_rows.append(save_method_metrics(fold_index, "biomedclip_mlp_14", test_mlp_metrics_tuned, fold_dirs, "val_tuned_f1"))
    summary_rows[-1].update(extra_tuned)
    label_fixed = test_mlp_metrics.copy(); label_fixed.insert(0, "threshold_mode", "fixed_0.5"); label_fixed.insert(0, "method", "biomedclip_mlp_14"); label_fixed.insert(0, "fold", fold_index); label_tables.append(label_fixed)
    label_tuned = test_mlp_metrics_tuned.copy(); label_tuned.insert(0, "threshold_mode", "val_tuned_f1"); label_tuned.insert(0, "method", "biomedclip_mlp_14"); label_tuned.insert(0, "fold", fold_index); label_tables.append(label_tuned)
    if SAVE_PREDICTIONS:
        save_predictions(fold_dirs["predictions"] / "biomedclip_mlp_14_test_predictions.csv", test, test_mlp_probs, test_y, "mlp")

    summary_df = pd.DataFrame(summary_rows)
    label_df = pd.concat(label_tables, ignore_index=True)
    summary_df.to_csv(fold_dirs["metrics"] / "fold_method_summary.csv", index=False)
    label_df.to_csv(fold_dirs["metrics"] / "fold_label_metrics.csv", index=False)
    save_json({
        "fold": int(fold_index),
        "status": "complete",
        "summary_path": str(fold_dirs["metrics"] / "fold_method_summary.csv"),
        "label_metrics_path": str(fold_dirs["metrics"] / "fold_label_metrics.csv"),
    }, fold_dirs["fold"] / "FOLD_COMPLETE.json")
    return {"summary": summary_df, "label_metrics": label_df, "training_logs": train_log}

## 8. Run 10-Fold Cross Validation

In [9]:
all_summary_frames = []
all_label_frames = []
all_training_logs = []

manifest = {
    "seed": SEED,
    "split_csv": str(SPLIT_CSV),
    "biomedclip_repo": BIOMEDCLIP_REPO,
    "methods": ["biomedclip_zero_shot", "biomedclip_mlp_14"],
    "fold_indices": FOLD_INDICES,
    "val_frac": VAL_FRAC,
    "mlp_hidden_dims": list(MLP_HIDDEN_DIMS),
    "mlp_epochs": MLP_EPOCHS,
    "output_root": str(OUTPUT_ROOT),
}
save_json(manifest, OUTPUT_ROOT / "experiment_manifest.json")

for fold_index in FOLD_INDICES:
    result = run_one_fold(fold_index)
    all_summary_frames.append(result["summary"])
    all_label_frames.append(result["label_metrics"])
    if len(result.get("training_logs", pd.DataFrame())):
        all_training_logs.append(result["training_logs"])
    pd.concat(all_summary_frames, ignore_index=True).to_csv(OUTPUT_ROOT / "biomedclip_10fold_method_summary_progress.csv", index=False)

summary_df = pd.concat(all_summary_frames, ignore_index=True)
label_df = pd.concat(all_label_frames, ignore_index=True)
summary_df.to_csv(OUTPUT_ROOT / "biomedclip_10fold_method_summary.csv", index=False)
label_df.to_csv(OUTPUT_ROOT / "biomedclip_10fold_label_metrics.csv", index=False)
if all_training_logs:
    pd.concat(all_training_logs, ignore_index=True).to_csv(OUTPUT_ROOT / "biomedclip_mlp_14_training_logs.csv", index=False)

display(summary_df)


===== Fold 0 =====
train_full 39690 train_part 35721 val 3969 test 4410
fold=0 epoch=0 loss=1.0464 val_disease_macro_auroc=0.7868
fold=0 epoch=1 loss=0.9094 val_disease_macro_auroc=0.8093
fold=0 epoch=2 loss=0.8758 val_disease_macro_auroc=0.8168
fold=0 epoch=3 loss=0.8614 val_disease_macro_auroc=0.8199
fold=0 epoch=4 loss=0.8561 val_disease_macro_auroc=0.8228
fold=0 epoch=5 loss=0.8504 val_disease_macro_auroc=0.8244
fold=0 epoch=6 loss=0.8458 val_disease_macro_auroc=0.8262
fold=0 epoch=7 loss=0.8424 val_disease_macro_auroc=0.8269
fold=0 epoch=8 loss=0.8400 val_disease_macro_auroc=0.8274
fold=0 epoch=9 loss=0.8390 val_disease_macro_auroc=0.8289
fold=0 epoch=10 loss=0.8356 val_disease_macro_auroc=0.8296
fold=0 epoch=11 loss=0.8336 val_disease_macro_auroc=0.8293
fold=0 epoch=12 loss=0.8322 val_disease_macro_auroc=0.8295
fold=0 epoch=13 loss=0.8304 val_disease_macro_auroc=0.8302
fold=0 epoch=14 loss=0.8298 val_disease_macro_auroc=0.8309
fold=0 epoch=15 loss=0.8269 val_disease_macro_auroc=

,fold,method,disease_macro_auroc,disease_macro_auprc,disease_macro_f1,all_macro_auroc,all_macro_auprc,all_macro_f1,num_scored_labels,threshold_mode,metrics_path,model_path,best_epoch,best_val_disease_macro_auroc
0,0,biomedclip_zero_shot,0.737865,0.151207,0.171377,0.741496,0.201588,0.218405,14,fixed_0.5,/data/liangz2/openi/multi_kg/biomedclip_10fold...,NaN,NaN,NaN
1,0,biomedclip_zero_shot,0.737865,0.151207,0.198972,0.741496,0.201588,0.243912,14,val_tuned_f1,/data/liangz2/openi/multi_kg/biomedclip_10fold...,NaN,NaN,NaN
2,0,biomedclip_mlp_14,0.814746,0.234566,0.224383,0.817429,0.282224,0.269551,14,fixed_0.5,/data/liangz2/openi/multi_kg/biomedclip_10fold...,/data/liangz2/openi/multi_kg/biomedclip_10fold...,24.0,0.833721
3,0,biomedclip_mlp_14,0.814746,0.234566,0.286425,0.817429,0.282224,0.327292,14,val_tuned_f1,/data/liangz2/openi/multi_kg/biomedclip_10fold...,/data/liangz2/openi/multi_kg/biomedclip_10fold...,24.0,0.833721
4,1,biomedclip_zero_shot,0.735519,0.160796,0.185881,0.739236,0.209825,0.232095,14,fixed_0.5,/data/liangz2/openi/multi_kg/biomedclip_10fold...,NaN,NaN,NaN
5,1,biomedclip_zero_shot,0.735519,0.160796,0.207124,0.739236,0.209825,0.251825,14,val_tuned_f1,/data/liangz2/openi/multi_kg/biomedclip_10fold...,NaN,NaN,NaN
6,1,biomedclip_mlp_14,0.821993,0.249005,0.221382,0.824391,0.295469,0.266763,14,fixed_0.5,/data/liangz2/openi/multi_kg/biomedclip_10fold...,/data/liangz2/openi/multi_kg/biomedclip_10fold...,28.0,0.831990
7,1,biomedclip_mlp_14,0.821993,0.249005,0.282723,0.824391,0.295469,0.324016,14,val_tuned_f1,/data/liangz2/openi/multi_kg/biomedclip_10fold...,/data/liangz2/openi/multi_kg/biomedclip_10fold...,28.0,0.831990
8,2,biomedclip_zero_shot,0.721736,0.157815,0.183825,0.726007,0.207252,0.229832,14,fixed_0.5,/data/liangz2/openi/multi_kg/biomedclip_10fold...,NaN,NaN,NaN
9,2,biomedclip_zero_shot,0.721736,0.157815,0.201835,0.726007,0.207252,0.246505,14,val_tuned_f1,/data/liangz2/openi/multi_kg/biomedclip_10fold...,NaN,NaN,NaN


## 9. Aggregate Paper Tables

In [ ]:
summary_path = OUTPUT_ROOT / "biomedclip_10fold_method_summary.csv"
label_path = OUTPUT_ROOT / "biomedclip_10fold_label_metrics.csv"
if not summary_path.exists():
    fold_summaries = sorted(OUTPUT_ROOT.glob("fold*/metrics/fold_method_summary.csv"))
    if not fold_summaries:
        raise FileNotFoundError("No fold summaries found. Run at least one fold first.")
    summary_df = pd.concat([pd.read_csv(p) for p in fold_summaries], ignore_index=True)
else:
    summary_df = pd.read_csv(summary_path)

metric_cols = [
    "disease_macro_auroc", "disease_macro_auprc", "disease_macro_f1",
    "all_macro_auroc", "all_macro_auprc", "all_macro_f1",
]
rows = []
for (method, threshold_mode), group in summary_df.groupby(["method", "threshold_mode"], sort=True):
    row = {
        "method": method,
        "threshold_mode": threshold_mode,
        "n_folds": int(group["fold"].nunique()),
        "completed_folds": ",".join(str(int(x)) for x in sorted(group["fold"].unique())),
    }
    for metric in metric_cols:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=1)
    rows.append(row)
aggregate_df = pd.DataFrame(rows)
aggregate_path = OUTPUT_ROOT / "biomedclip_10fold_aggregate_summary.csv"
aggregate_df.to_csv(aggregate_path, index=False)
print("Saved", aggregate_path)
display(aggregate_df.sort_values(["threshold_mode", "disease_macro_auroc_mean"], ascending=[True, False]))

if label_path.exists():
    label_df = pd.read_csv(label_path)
else:
    fold_labels = sorted(OUTPUT_ROOT.glob("fold*/metrics/fold_label_metrics.csv"))
    label_df = pd.concat([pd.read_csv(p) for p in fold_labels], ignore_index=True)
label_agg = label_df.groupby(["method", "threshold_mode", "label"], as_index=False).agg(
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
    auprc_mean=("auprc", "mean"), auprc_std=("auprc", "std"),
    f1_mean=("f1", "mean"), f1_std=("f1", "std"),
    support_mean=("support", "mean"),
)
label_agg_path = OUTPUT_ROOT / "biomedclip_10fold_label_aggregate_summary.csv"
label_agg.to_csv(label_agg_path, index=False)
print("Saved", label_agg_path)
display(label_agg.head())

## Notes

Use `fixed_0.5` for a consistent operating-point comparison and AUROC/AUPRC for ranking performance. Use `val_tuned_f1` when reporting threshold-calibrated F1; AUROC and AUPRC do not change under threshold tuning.